# 03 — PD Train/Test Split

This notebook creates reproducible training and test datasets for the Probability of Default (PD) model.

The split is performed **before** any transformation that learns information from the data, including:

- statistical imputation;
- automated binning;
- Weight of Evidence calculation;
- Information Value calculation;
- feature selection;
- scaling or standardisation.

Target convention:

- `good_bad = 0`: bad borrower;
- `good_bad = 1`: good borrower.

A stratified split is used to preserve approximately the same good/bad proportions in both datasets.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


## 1. Resolve project paths


In [2]:
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SPLIT_DIR = PROCESSED_DIR / "pd_split"
REPORT_DIR = PROJECT_ROOT / "reports" / "pd_split"

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Processed data directory: {PROCESSED_DIR.resolve()}")
print(f"Split output directory: {SPLIT_DIR.resolve()}")
print(f"Report directory: {REPORT_DIR.resolve()}")


Project root: C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new
Processed data directory: C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new\data\processed
Split output directory: C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new\data\processed\pd_split
Report directory: C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new\reports\pd_split


## 2. Load the PD modelling dataset

Parquet is preferred because it preserves data types. CSV is used as a fallback.


In [3]:
PD_PARQUET_PATH = PROCESSED_DIR / "pd_modeling_dataset.parquet"
PD_CSV_PATH = PROCESSED_DIR / "pd_modeling_dataset.csv"

if PD_PARQUET_PATH.exists():
    pd_data = pd.read_parquet(PD_PARQUET_PATH)
    source_path = PD_PARQUET_PATH
elif PD_CSV_PATH.exists():
    pd_data = pd.read_csv(PD_CSV_PATH, low_memory=False)
    source_path = PD_CSV_PATH
else:
    raise FileNotFoundError(
        "PD modelling dataset not found. Run 02_PD_Dataset_Construction.ipynb first."
    )

print(f"Loaded: {source_path.resolve()}")
print(f"Shape: {pd_data.shape}")
display(pd_data.head())


Loaded: C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new\data\processed\pd_modeling_dataset.parquet
Shape: (466285, 49)


,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,home_ownership,annual_inc,verification_status,issue_d,purpose,addr_state,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,emp_length_missing,emp_length_years,credit_history_months,funding_ratio,investor_funding_ratio,investor_loan_ratio,loan_to_income_ratio,installment_to_income_ratio,open_account_ratio,credit_inquiry_rate,delinquency_rate,emp_title_missing,loan_burden_interest,mths_since_last_record_is_missing,mths_since_last_major_derog_is_missing,mths_since_last_delinq_is_missing,open_account_inconsistency,good_bad
0,5000,5000,4975.0,36,10.65,162.87,B,RENT,24000.0,Verified,2011-12-01,credit_card,AZ,27.65,0.0,1985-01-01,1.0,-1.0,-1.0,3.0,0.0,13648,83.7,9.0,f,0.0,-1.0,0.0,NaN,NaN,NaN,0,10,323,1.0,0.995,0.995,0.208333,0.081435,0.333333,0.037152,0.0,1,2.218750,1,1,1,0,1
1,2500,2500,2500.0,60,15.27,59.83,C,RENT,30000.0,Source Verified,2011-12-01,car,GA,1.00,0.0,1999-04-01,5.0,-1.0,-1.0,3.0,0.0,1687,9.4,4.0,f,0.0,-1.0,0.0,NaN,NaN,NaN,0,0,152,1.0,1.000,1.000,0.083333,0.023932,0.750000,0.394737,0.0,0,1.272500,1,1,1,0,0
2,2400,2400,2400.0,36,15.96,84.33,C,RENT,12252.0,Not Verified,2011-12-01,small_business,IL,8.72,0.0,2001-11-01,2.0,-1.0,-1.0,2.0,0.0,2956,98.5,10.0,f,0.0,-1.0,0.0,NaN,NaN,NaN,0,10,121,1.0,1.000,1.000,0.195886,0.082595,0.200000,0.198347,0.0,1,3.126347,1,1,1,0,1
3,10000,10000,10000.0,36,13.49,339.31,C,RENT,49200.0,Source Verified,2011-12-01,other,CA,20.00,0.0,1996-02-01,1.0,35.0,-1.0,10.0,0.0,5598,21.0,37.0,f,0.0,-1.0,0.0,NaN,NaN,NaN,0,10,190,1.0,1.000,1.000,0.203252,0.082759,0.270270,0.063158,0.0,0,2.741870,1,1,0,0,1
4,3000,3000,3000.0,60,12.69,67.79,B,RENT,80000.0,Source Verified,2011-12-01,other,OR,17.94,0.0,1996-01-01,0.0,38.0,-1.0,15.0,0.0,27783,53.9,38.0,f,0.0,-1.0,0.0,NaN,NaN,NaN,0,1,191,1.0,1.000,1.000,0.037500,0.010169,0.394737,0.0,0.0,0,0.475875,1,1,0,0,1


## 3. Validate the target and dataset structure


In [4]:
TARGET_COLUMN = "good_bad"

if TARGET_COLUMN not in pd_data.columns:
    raise KeyError(f"Missing target column: {TARGET_COLUMN}")

duplicate_columns = pd_data.columns[pd_data.columns.duplicated()].tolist()
unexpected_target_values = sorted(
    set(pd_data[TARGET_COLUMN].dropna().unique()) - {0, 1}
)

assert not duplicate_columns, f"Duplicate column names detected: {duplicate_columns}"
assert pd_data[TARGET_COLUMN].isna().sum() == 0, "The target contains missing values."
assert not unexpected_target_values, f"Unexpected target values: {unexpected_target_values}"
assert pd_data[TARGET_COLUMN].nunique() == 2, "Both target classes must be present."

target_distribution = (
    pd_data[TARGET_COLUMN]
    .value_counts()
    .sort_index()
    .rename_axis(TARGET_COLUMN)
    .to_frame("record_count")
)

target_distribution["class_label"] = target_distribution.index.map({
    0: "Bad",
    1: "Good",
})

target_distribution["percentage"] = (
    target_distribution["record_count"] / len(pd_data) * 100
)

display(target_distribution)


,record_count,class_label,percentage
good_bad,,,
0,50968,Bad,10.930654
1,415317,Good,89.069346


## 4. Separate predictors and target

No transformations are applied at this stage.


In [5]:
X = pd_data.drop(columns=[TARGET_COLUMN]).copy()
y = pd_data[TARGET_COLUMN].copy()

assert TARGET_COLUMN not in X.columns
assert len(X) == len(y)
assert X.index.equals(y.index)

print(f"Predictor matrix shape: {X.shape}")
print(f"Target vector shape:    {y.shape}")


Predictor matrix shape: (466285, 48)
Target vector shape:    (466285,)


## 5. Define split configuration

The configuration is kept explicit so the split can be reproduced exactly.


In [6]:
TEST_SIZE = 0.20
RANDOM_STATE = 42

print(f"Test size: {TEST_SIZE:.0%}")
print(f"Training size: {1 - TEST_SIZE:.0%}")
print(f"Random state: {RANDOM_STATE}")
print("Stratification: enabled")


Test size: 20%
Training size: 80%
Random state: 42
Stratification: enabled


## 6. Perform the stratified train/test split


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")


X_train shape: (373028, 48)
X_test shape:  (93257, 48)
y_train shape: (373028,)
y_test shape:  (93257,)


## 7. Validate split integrity

The checks below confirm that train and test contain no overlapping observations, all rows are accounted for, predictors and targets remain aligned, and both classes are present in each subset.


In [8]:
train_test_index_overlap = X_train.index.intersection(X_test.index)

assert len(train_test_index_overlap) == 0, "Train and test sets overlap."
assert len(X_train) + len(X_test) == len(X), "Some observations are missing after the split."
assert X_train.index.equals(y_train.index), "X_train and y_train are not aligned."
assert X_test.index.equals(y_test.index), "X_test and y_test are not aligned."
assert y_train.nunique() == 2, "Training target does not contain both classes."
assert y_test.nunique() == 2, "Test target does not contain both classes."

print("Split integrity checks passed.")


Split integrity checks passed.


## 8. Compare target distributions


In [9]:
def target_summary(target: pd.Series, dataset_name: str) -> pd.DataFrame:
    summary = (
        target.value_counts()
        .sort_index()
        .rename_axis("good_bad")
        .to_frame("record_count")
    )
    summary["percentage"] = summary["record_count"] / len(target) * 100
    summary["dataset"] = dataset_name
    summary["class_label"] = summary.index.map({0: "Bad", 1: "Good"})
    return summary.reset_index()

distribution_comparison = pd.concat(
    [
        target_summary(y, "Full dataset"),
        target_summary(y_train, "Train"),
        target_summary(y_test, "Test"),
    ],
    ignore_index=True,
)

display(distribution_comparison)

print(f"Full bad rate:  {(y == 0).mean():.4%}")
print(f"Train bad rate: {(y_train == 0).mean():.4%}")
print(f"Test bad rate:  {(y_test == 0).mean():.4%}")


,good_bad,record_count,percentage,dataset,class_label
0,0,50968,10.930654,Full dataset,Bad
1,1,415317,89.069346,Full dataset,Good
2,0,40774,10.930547,Train,Bad
3,1,332254,89.069453,Train,Good
4,0,10194,10.931083,Test,Bad
5,1,83063,89.068917,Test,Good


Full bad rate:  10.9307%
Train bad rate: 10.9305%
Test bad rate:  10.9311%


## 9. Verify that no preprocessing has leaked across the split

This notebook intentionally leaves missing values, categories and numerical scales unchanged. Binning and WoE rules will be learned only from the training set in the next stage.


In [10]:
split_quality_summary = pd.DataFrame({
    "dataset": ["X_train", "X_test"],
    "rows": [len(X_train), len(X_test)],
    "columns": [X_train.shape[1], X_test.shape[1]],
    "missing_values": [
        int(X_train.isna().sum().sum()),
        int(X_test.isna().sum().sum()),
    ],
    "columns_with_missing_values": [
        int((X_train.isna().sum() > 0).sum()),
        int((X_test.isna().sum() > 0).sum()),
    ],
    "numeric_columns": [
        X_train.select_dtypes(include=[np.number]).shape[1],
        X_test.select_dtypes(include=[np.number]).shape[1],
    ],
    "non_numeric_columns": [
        X_train.select_dtypes(exclude=[np.number]).shape[1],
        X_test.select_dtypes(exclude=[np.number]).shape[1],
    ],
})

display(split_quality_summary)


,dataset,rows,columns,missing_values,columns_with_missing_values,numeric_columns,non_numeric_columns
0,X_train,373028,48,185938,21,40,8
1,X_test,93257,48,46718,17,40,8


## 10. Prepare exports

Original row indices are preserved in `source_row_index` for traceability.


In [11]:
X_train_export = X_train.copy()
X_test_export = X_test.copy()

X_train_export.insert(0, "source_row_index", X_train_export.index)
X_test_export.insert(0, "source_row_index", X_test_export.index)

y_train_export = y_train.rename(TARGET_COLUMN).to_frame()
y_test_export = y_test.rename(TARGET_COLUMN).to_frame()

y_train_export.insert(0, "source_row_index", y_train_export.index)
y_test_export.insert(0, "source_row_index", y_test_export.index)

X_train_export = X_train_export.reset_index(drop=True)
X_test_export = X_test_export.reset_index(drop=True)
y_train_export = y_train_export.reset_index(drop=True)
y_test_export = y_test_export.reset_index(drop=True)

display(X_train_export.head())
display(y_train_export.head())


,source_row_index,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,home_ownership,annual_inc,verification_status,issue_d,purpose,addr_state,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,emp_length_missing,emp_length_years,credit_history_months,funding_ratio,investor_funding_ratio,investor_loan_ratio,loan_to_income_ratio,installment_to_income_ratio,open_account_ratio,credit_inquiry_rate,delinquency_rate,emp_title_missing,loan_burden_interest,mths_since_last_record_is_missing,mths_since_last_major_derog_is_missing,mths_since_last_delinq_is_missing,open_account_inconsistency
0,456615,15000,15000,15000.0,36,8.90,476.30,A,MORTGAGE,80000.0,Source Verified,2014-01-01,credit_card,WI,17.01,1.0,1995-12-01,0.0,20.0,-1.0,19.0,0.0,20699,59.0,32.0,w,0.0,-1.0,0.0,0.0,143586.0,35100.0,0,7,217,1.0,1.000000,1.000000,0.187500,0.071445,0.593750,0.0,0.031250,0,1.668750,1,1,0,0
1,451541,8000,8000,8000.0,60,18.25,204.24,D,OWN,44000.0,Verified,2014-01-01,other,TN,23.46,0.0,1995-07-01,1.0,-1.0,-1.0,12.0,0.0,13245,32.1,25.0,f,0.0,-1.0,0.0,0.0,180443.0,41300.0,0,10,222,1.0,1.000000,1.000000,0.181818,0.055702,0.480000,0.054054,0.000000,0,3.318182,1,1,1,0
2,394474,12150,12150,12100.0,60,18.92,314.65,D,OWN,27000.0,Source Verified,2014-05-01,credit_card,TN,31.07,1.0,1990-09-01,0.0,10.0,-1.0,9.0,0.0,7172,73.2,22.0,f,0.0,-1.0,0.0,0.0,34197.0,9800.0,0,3,284,1.0,0.995885,0.995885,0.450000,0.139844,0.409091,0.0,0.045455,0,8.514000,1,1,0,0
3,110294,10000,10000,10000.0,36,6.03,304.36,A,MORTGAGE,33000.0,Not Verified,2013-08-01,debt_consolidation,TN,9.16,0.0,2001-12-01,0.0,-1.0,-1.0,5.0,0.0,2138,21.2,17.0,w,0.0,-1.0,0.0,0.0,77959.0,10100.0,0,3,140,1.0,1.000000,1.000000,0.303030,0.110676,0.294118,0.0,0.000000,0,1.827273,1,1,1,0
4,139343,15825,15825,15825.0,36,12.12,526.53,B,MORTGAGE,59000.0,Verified,2013-05-01,debt_consolidation,WA,17.94,0.0,1979-06-01,0.0,-1.0,-1.0,7.0,0.0,30326,93.6,31.0,f,0.0,-1.0,0.0,0.0,187370.0,32400.0,0,10,407,1.0,1.000000,1.000000,0.268220,0.107091,0.225806,0.0,0.000000,0,3.250831,1,1,1,0


,source_row_index,good_bad
0,456615,1
1,451541,1
2,394474,1
3,110294,1
4,139343,0


## 11. Export split datasets and reports


In [12]:
export_objects = {
    "X_train": X_train_export,
    "X_test": X_test_export,
    "y_train": y_train_export,
    "y_test": y_test_export,
}

parquet_exported = True

try:
    for name, dataframe in export_objects.items():
        dataframe.to_parquet(SPLIT_DIR / f"{name}.parquet", index=False)
except ImportError as error:
    parquet_exported = False
    print("Parquet export skipped. Install pyarrow or fastparquet to enable it.")
    print(f"Details: {error}")

for name, dataframe in export_objects.items():
    dataframe.to_csv(SPLIT_DIR / f"{name}.csv", index=False)

distribution_report_path = REPORT_DIR / "target_distribution_comparison.csv"
quality_report_path = REPORT_DIR / "split_quality_summary.csv"
configuration_report_path = REPORT_DIR / "split_configuration.csv"

distribution_comparison.to_csv(distribution_report_path, index=False)
split_quality_summary.to_csv(quality_report_path, index=False)

split_configuration = pd.DataFrame({
    "parameter": [
        "target_column",
        "good_class",
        "bad_class",
        "test_size",
        "train_size",
        "random_state",
        "stratified",
    ],
    "value": [
        TARGET_COLUMN,
        1,
        0,
        TEST_SIZE,
        1 - TEST_SIZE,
        RANDOM_STATE,
        True,
    ],
})

split_configuration.to_csv(configuration_report_path, index=False)

print("=" * 70)
print("PD TRAIN/TEST SPLIT EXPORT COMPLETED")
print("=" * 70)

for name in export_objects:
    if parquet_exported:
        print(f"{name} (Parquet): {(SPLIT_DIR / f'{name}.parquet').resolve()}")
    print(f"{name} (CSV):     {(SPLIT_DIR / f'{name}.csv').resolve()}")

print(f"Distribution report: {distribution_report_path.resolve()}")
print(f"Quality report:      {quality_report_path.resolve()}")
print(f"Configuration:       {configuration_report_path.resolve()}")


PD TRAIN/TEST SPLIT EXPORT COMPLETED
X_train (Parquet): C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new\data\processed\pd_split\X_train.parquet
X_train (CSV):     C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new\data\processed\pd_split\X_train.csv
X_test (Parquet): C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new\data\processed\pd_split\X_test.parquet
X_test (CSV):     C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new\data\processed\pd_split\X_test.csv
y_train (Parquet): C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new\data\processed\pd_split\y_train.parquet
y_train (CSV):     C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new\data\processed\pd_split\y_train.csv
y_test (Parquet): C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new\data\processed\pd_split\y_test.parquet
y_test (CSV):     C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-sc

## Next step

The next notebook will build initial bins **using the training set only**.

Recommended sequence:

1. join `X_train` and `y_train` using `source_row_index`;
2. inspect candidate variables and cardinalities;
3. generate initial bins with `scorecardpy.woebin`;
4. review WoE trends, sample sizes and default behaviour;
5. manually adjust important variables when needed;
6. save the final binning rules;
7. apply the same bins to the untouched test set.
